# Expfit notes 4b: Special forms for multiple exponentials

When fitting more than one exponential, we want to restrict the values parameters can take.

1. With two growing exponentials, one will rapidly make the other unobservable, so beyond the single exponential case we want $c < 0$.
2. For the common use case of bi-exponential decay (or tri-exponential etc.), we want all $b > 0$ or all $b < 0$
3. Another common case is to have $m^+$ rapid exponentials in one direction, followed by $m^-$ slow exponentials in the other (e.g. $b_{i < m^+}>0,\ b_{i \geq m^+} < 0,\ c_i < 0$).

To achieve this without modifying the optimiser, we can use log transformed parameters.

## Restricting $c$

We set $\gamma=\log(-c) \rightarrow c=-e^\gamma$ and redefine the MSE as

\begin{align}
E_m &= \frac{1}{n}\sum_{i=1}^n \left( y - a - \sum_{j=1}^m b_j e^{-e^{q_j} x_i} \right)^2 &&  \\
\end{align}

The shorthand becomes
\begin{align}
e_j = e^{-e^{\gamma_j} x_i}
\end{align}
with partial derivative
\begin{align}
\frac{\partial}{\partial \gamma_j} e_j = -x_i e_j e^{\gamma_j}
\end{align}

Compared to $\frac{\partial}{\partial c_j} e^{c_j x_i} = x_i e^{c_j x_i}$, this means an extra multiplier $-e^\gamma$.

### Jacobian

\begin{align}
\frac{\partial E_m}{\partial a}   &= \frac{2}{n}\sum_{i=1}^n f \\
\frac{\partial E_m}{\partial b_s} &= \frac{2}{n}\sum_{i=1}^n f e_s \\
\frac{\partial E_m}{\partial c_s} &= -\frac{2b_s}{n}\sum_{i=1}^n f x e_s e^{\gamma_s}
\end{align}

### Hessian

For $a$
\begin{align}
\frac{\partial^2 E_2}{\partial a^2} &= 2 \\
\frac{\partial^2 E_2}{\partial b_r\,\partial a} &= \frac{2}{n}\sum^n e_r \\
\frac{\partial^2 E_2}{\partial \gamma_r\,\partial a} &= -\frac{2b_r}{n}\sum^n x e_s e^{\gamma_s}
\end{align}

For $b$
\begin{align}
\frac{\partial^2 E_2}{\partial b_r\,\partial b_s} &= \frac{2}{n}\sum_{i=1}^n e_r e_s \\
\end{align}

For $b$ and $\gamma$
\begin{align}
\frac{\partial^2 E_2}{\partial \gamma_s\,\partial b_s} &= -\frac{2}{n}\sum_{i=1}^n (f + b_s e_s) xe_se^{\gamma_s} \\
r \neq s, \frac{\partial^2 E_2}{\partial \gamma_r\,\partial b_s} &= -\frac{2b_r}{n}\sum_{i=1}^n x e_r e_s e^{\gamma_r}
\end{align}

For $\gamma$
\begin{align}
r \neq s, \frac{\partial^2 E_2}{\partial \gamma_r \gamma_s} &= \frac{2b_rb_s}{n}\sum_{i=1}^n x^2 e_r e_s e^{\gamma_r} e^{\gamma_s} \\
\frac{\partial^2 E_2}{\partial \gamma_s^2} &= \frac{2b_s}{n}\sum_{i=1}^n 
\left( (f + b_se_s) x^2 e^{2\gamma_s} e_s - f_sxe_s e^{q_s} \right)
\end{align}

The final equation is more complicated than before, as it involves the derivative of $e_s e^{\gamma_s}$, which is a product of two functions of $\gamma_s$.

## Checking against finite differences

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
x = np.linspace(0, 1, 100)

a, b, c = 5, 5, -3
y = a + b * np.exp(-c * x)

a0, b0, c0 = a + 1, b - 1, c + 1
y0 = a + b * np.exp(-c * x)

ct0 = np.log(-c0)

In [3]:
def mse(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2
    p = np.asarray(p)    
    bs = p[1::2].reshape((m, 1))
    cs = -np.exp(p[2::2].reshape((m, 1)))
    return np.sum((p[0] - y + np.sum(bs * np.exp(cs * x), axis=0))**2) / len(x)


def mse_jac_fd(x, y, p, dp):
    e = mse(x, y, p)
    jac = np.zeros(len(p))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        jac[i] = (mse(x, y, q) - e) / dp
    return e, jac
    
    
def mse_jac_hes_fd(x, y, p, dp=1e-5):
    d = len(p)
    m = (d - 1) // 2
    mse, jac = mse_jac_fd(x, y, p, dp)
    
    hes = np.zeros((d, d))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        hes[i] = (mse_jac_fd(x, y, q, dp)[1] - jac) / dp   
    return mse, jac, hes


ERROR! Session/line number was not unique in database. History logging moved to new session 88


In [4]:
def mse_jac_hes(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2

    n = len(x)
    ninv2 = 2 / n
    
    # Unpack
    p = np.asarray(p)
    a = p[0]
    b = p[1::2].reshape((m, 1))       # (m, 1)
    ct = p[2::2].reshape((m, 1))      # (m, 1)
    
    # MSE
    ect = np.exp(ct)
    e = np.exp(-ect * x)            # (m, n)
    be = b * e                      # (m, n)
    f = a - y + np.sum(be, axis=0)  # (n, )
    mse = np.sum(f**2) / n

    # Jacobian
    ex = e * x * ect
    jac = np.zeros(d)
    jac[0] = ninv2 * np.sum(f)
    jac[1::2] = ninv2 * np.sum(f * e, axis=1)
    jac[2::2] = -ninv2 * np.sum(f * ex, axis=1) * b.T

    # Hessian
    hes = np.zeros((d, d))
    # aa, ab, ac
    hes[0, 0] = 2
    hes[0, 1::2] = hes[1::2, 0] = ninv2 * np.sum(e, axis=1)
    hes[0, 2::2] = hes[2::2, 0] = -ninv2 * np.sum(ex, axis=1) * b.T
    for i in range(m):
        # bi^2, ci^2, and bi*ci
        hes[1 + 2 * i, 1 + 2 * i] = ninv2 * np.sum(e[i]**2)       
        hes[2 + 2 * i, 2 + 2 * i] = ninv2 * b[i, 0] * np.sum(
            ((f + be[i]) * x * ect[i] - f) * ex[i])
        hes[1 + 2 * i, 2 + 2 * i] = hes[2 + 2 * i, 1 + 2 * i] = \
            -ninv2 * np.sum((f + be[i]) * ex[i])
        for j in range(i + 1, m):
            # bi*bj, ci*cj, bi*cj, bj*ci
            hes[1 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 1 + 2 * i] = \
                ninv2 * np.sum(e[i] * e[j])
            hes[2 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 2 + 2 * i] = \
                ninv2 * np.sum(ex[i] * ex[j]) * b[i, 0] * b[j, 0]
            hes[1 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 1 + 2 * i] = \
                -ninv2 * np.sum(ex[j] * e[i]) * b[j, 0]
            hes[2 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 2 + 2 * i] = \
                -ninv2 * np.sum(ex[i] * e[j]) * b[i, 0]

    return mse, jac, hes

In [5]:
def mse_jac_hes(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2

    n = len(x)
    ninv2 = 2 / n
    
    # Unpack
    p = np.asarray(p)
    a = p[0]
    bs = p[1::2].reshape((m, 1))        # (m, 1)
    qs = p[2::2].reshape((m, 1))        # (m, 1)

    # MSE
    n = len(x)
    ninv2 = 2 / n
    eqs = np.exp(qs)
    es = np.exp(-eqs * x)
    bes = bs * es
    fs = a - y + np.sum(bes, axis=0)
    mse = np.sum(fs**2) / n

    # Jacobian
    jac = np.zeros(d)
    xes = es * x * eqs
    jac[0] = ninv2 * np.sum(fs)
    jac[1::2] = ninv2 * np.sum(fs * es, axis=1)
    jac[2::2] = -ninv2 * np.sum(fs * xes, axis=1) * bs.T

    # Hessian
    hes = np.zeros((d, d))
    # aa, ab, ac
    hes[0, 0] = 2
    hes[0, 1::2] = hes[1::2, 0] = ninv2 * np.sum(es, axis=1)
    hes[0, 2::2] = hes[2::2, 0] = -ninv2 * np.sum(xes, axis=1) * bs.T
    for i in range(m):
        # bi^2, ci^2, and bi*ci
        hes[1 + 2 * i, 1 + 2 * i] = ninv2 * np.sum(es[i]**2)
        hes[2 + 2 * i, 2 + 2 * i] = ninv2 * bs[i, 0] * np.sum(
            ((fs + bes[i]) * x * eqs[i] - fs) * xes[i])
        hes[1 + 2 * i, 2 + 2 * i] = hes[2 + 2 * i, 1 + 2 * i] = \
            -ninv2 * np.sum((fs + bes[i]) * xes[i])
        for j in range(i + 1, m):
            # bi*bj, ci*cj, bi*cj, bj*ci
            hes[1 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 1 + 2 * i] = \
                ninv2 * np.sum(es[i] * es[j])
            hes[2 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 2 + 2 * i] = \
                ninv2 * np.sum(xes[i] * xes[j]) * bs[i, 0] * bs[j, 0]
            hes[1 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 1 + 2 * i] = \
                -ninv2 * np.sum(xes[j] * es[i]) * bs[j, 0]
            hes[2 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 2 + 2 * i] = \
                -ninv2 * np.sum(xes[i] * es[j]) * bs[i, 0]
            
    return mse, jac, hes


And now we can do a proper comparison:

In [6]:
m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, ct0, 3, 0.5))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, ct0, 3, 0.5))
print(m1)
print(m2)
print()
print(j1)
print(j2)
print()
with np.printoptions(linewidth=120):
    print(h1)
    print(h2)
    print()
    print(np.abs(h1 - h2))

1535.1084253981558
1535.1084253981558

[-55.62593905 -12.72835526  71.1813801  -16.31081805  57.51408267]
[-55.62592908 -12.72835279  71.18121325 -16.31081514  57.51401475]

[[  2.           0.86740054  -2.36289075   0.98194234  -1.77717235]
 [  0.86740054   0.49618302  16.89468426   0.53885365  -0.647821  ]
 [ -2.36289075  16.89468426 -33.36587882  -1.04779546   2.28696421]
 [  0.98194234   0.53885365  -1.04779546   0.58868481  18.41205341]
 [ -1.77717235  -0.647821     2.28696421  18.41205341 -13.5707412 ]]
[[  2.00316208   0.86856744  -2.36468622   0.98452801  -1.77351467]
 [  0.86856744   0.49567461  16.89159035   0.54342308  -0.64119376]
 [ -2.36468622  16.89159035 -33.36708687  -1.04591891   2.29420039]
 [  0.98452801   0.54342308  -1.04591891   0.58889782  18.41954145]
 [ -1.77351467  -0.64119376   2.29420039  18.41954145 -13.56283974]]

[[0.00316208 0.0011669  0.00179547 0.00258568 0.00365768]
 [0.0011669  0.00050841 0.00309391 0.00456943 0.00662723]
 [0.00179547 0.00309391 0.0

This looks OK

In [7]:
m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, ct0, 3, -7, 1.1, 2.2))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, ct0, 3, -7, 1.1, 2.2))
print(m1)
print(m2)
print()
with np.printoptions(linewidth=120):
    print(j1)
    print(j2)
    print()
    print(np.abs(h1 - h2) < 5e-5)

1420.1457304465498
1420.1457304465498

[-52.32203222 -11.53477055  67.09645278 -52.28457445   0.11233122   0.11929354   0.98618367]
[-52.32202225 -11.53476808  67.0962906  -52.28456446   0.1123318    0.11929415   0.98616599]

[[False False False False False False False]
 [False False False False False False False]
 [False False False False False False False]
 [False False False False False False False]
 [False False False False False False False]
 [False False False False False False False]
 [False False False False False False False]]
